In [ ]:
%load_ext autoreload
%autoreload 2
import os
print(os.getcwd())

/home/as/code/ai/susteelaible/nlp


# bert-1
- run pipeline

In [ ]:
from nlp import ClimateBERTAnalyzer, analyze_reports

stats = analyze_reports('../data/reports/Baosteel')
# stats = analyze_reports("../data/reports")

# bert-2
- run vizualisations

In [ ]:
from nlp import ClimateBERTVisualizer, visualize_results

visualize_results("../cache", "../out")

✅ Loaded: 15 reports, 1 companies, 2013-2020

EXPORTING CSV FILES
   ✓ company_year.csv (8 rows)
   ✓ company_totals.csv (1 companies)
   ✓ yearly_industry.csv (8 years)
   ✓ funnel_company_year.csv (8 rows)

   📁 All CSVs saved to: ../out/

GENERATING PLOTS
   ✓ slide_main.png
   ✓ slide_sentiment_trend.png
   ✓ talk_score_trend.png
   ✓ funnel_trend.png
   ✓ talk_score_per_company.png
   ✓ per_company_components.png
   ✓ per_company_sentiment.png
   ✓ sentiment_all_companies.png
   ✓ n0_funnel.png
   ✓ n0_quality_comparison.png
   ✓ n0_per_company.png
   ✓ n0_gap_analysis.png

   📁 All plots saved to: ../out/

   Generating word frequency plots...

   📊 ALL CHUNKS (top 30):
   environment(1232), iron(766), management(736), development(720), energy(628), technology(609), products(607), protection(559), production(525), industry(522), green(473), emission(453), system(428), reduction(421), base(410)

   🌱 OPPORTUNITY chunks (top 15):
   environment(809), development(574), technology(52

# rag 1

## LLM Model Selection

Before running extraction, test which Ollama model works best for your GPU. Key criteria:
- **Speed**: Should be 1-5s per call (not 100s+)
- **Format compliance**: Must output exact `[chunk_id]|||text` format
- **Accuracy**: Should copy text verbatim, not paraphrase

| Model | Speed | Format | Notes |
|-------|-------|--------|-------|
| `gemma3:4b` | ~4s | ✅ Correct | **Recommended** - fast, follows instructions |
| `phi3:mini` | ~3s | ❌ Mangles IDs | Fast but outputs `[Chunk-1]` instead of actual IDs |
| `qwen3:4b` | ~108s | ✅ Correct | Hidden "thinking" tokens, 25x slower |
| `llama3.1:8b` | ~3s | ? | too large for 4GB VRAM |
| `mistral:7b` | ~5s | ? | too large for 4GB VRAM  |

In [ ]:
# LLM Speed Test - compare models before running extraction
# Uncomment models to test. Expect: 1-5s = good, 100s+ = broken
import time
from langchain_ollama import ChatOllama

TEST_PROMPT = """Extract BARRIERS from this text.
OUTPUT FORMAT: [chunk_id]|||verbatim text
If none: NONE_FOUND

TEXT:
[012_001]
The high cost of green hydrogen limits decarbonisation options.

[012_002]
We committed to reducing emissions by 50% by 2030."""

def test_model(model, extra_opts=None):
    """Test a single model for speed and format compliance."""
    opts = {"temperature": 0, "num_ctx": 4096}
    if extra_opts:
        opts.update(extra_opts)

    llm = ChatOllama(model=model, **opts)

    # Warm-up (model loading)
    print(f"\n{model}:")
    start = time.time()
    llm.invoke("Hi")
    print(f"  Warm-up: {time.time() - start:.1f}s")

    # Extraction test
    start = time.time()
    resp = llm.invoke(TEST_PROMPT)
    elapsed = time.time() - start

    # Check format compliance
    has_correct_format = "[012_001]|||" in resp.content or "[012_002]|||" in resp.content
    format_status = "✅" if has_correct_format else "❌"

    print(f"  Time: {elapsed:.2f}s | Format: {format_status} | Chars: {len(resp.content)}")
    print(f"  Output: {resp.content[:120]}...")
    return elapsed, has_correct_format

# === MODELS TO TEST ===
# Uncomment models you have installed via `ollama pull <model>`
models_to_test = [
    "gemma3:4b",           # Recommended - fast + correct format
    # "gemma3:1b",         # Smaller, faster, may be less accurate
    # "llama3.2:3b",       # Good alternative
    "llama3.1:8b",       # Larger, needs more VRAM
    "phi3:mini",         # Fast but mangles chunk IDs
    # "mistral:7b",        # Larger, may be slower
    "qwen3:4b",          # SLOW - hidden thinking tokens, avoid
]

results = {}
for model in models_to_test:
    try:
        elapsed, correct = test_model(model)
        results[model] = (elapsed, correct)
    except Exception as e:
        print(f"\n{model}: ERROR - {e}")

print(f"\n{'='*50}")
print("SUMMARY (sorted by speed)")
print(f"{'='*50}")
for model, (t, correct) in sorted(results.items(), key=lambda x: x[1][0]):
    status = "✅ GOOD" if correct and t < 10 else "❌ AVOID"
    print(f"  {model}: {t:.1f}s {'✅' if correct else '❌'} {status}")

In [1]:
from nlp import quick_start, RAGConfig, analyze_token_usage

# Configure RAG pipeline parameters here
rag_config = RAGConfig(
    cache_dir="../cache",
    use_bert_cache=True,
    output_folder="../out",

    # LLM settings
    ollama_model="gemma3:4b",  # Fast + follows format. Avoid qwen3 (hidden thinking = slow)
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,
    llm_num_ctx=4096,  # Context window size

    # Extraction settings
    max_chunks_per_group=7,  # Limit chunks sent to LLM per company-year

    # BERT filtering (reduce chunk count by filtering low-confidence chunks)
    min_detector_score=0.7,  # 0.0 = no filter, 0.7 = keep only confident climate chunks
)

# Initialize pipeline and analyze token usage
pipeline = quick_start(config=rag_config)
stats = analyze_token_usage(pipeline)

# Auto-apply recommended max_chunks_per_group (optional)
rag_config.max_chunks_per_group = stats["recommendations"]["max_chunks_for_ctx"]
stats["recommendations"]["max_chunks_for_ctx"]

/home/as/code/ai/susteelaible/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Filtered 1048 chunks below detector_score=0.7 (14545 remaining)
('001', '2022'): 503 chunks
('001', '2023'): 487 chunks
('001', '2021'): 484 chunks
('015', '2024'): 344 chunks
('001', '2019'): 338 chunks
('001', '2024'): 336 chunks
('001', '2020'): 332 chunks
('003', '2024'): 279 chunks
('015', '2025'): 277 chunks
('001', '2013'): 262 chunks
✓ Loaded 14545 chunks from 15 companies (../cache)
PROMPT OVERHEAD
  BARRIER_MAP_PROMPT:   ~151 tokens (606 chars)
  MOTIVATOR_MAP_PROMPT: ~151 tokens (606 chars)
  Using max as overhead: 151 tokens

CHUNK-LEVEL STATISTICS
Total chunks: 14545

Chunk size (characters):
  Min: 38, Max: 19629, Mean: 1753, Median: 1453

Chunk size (tokens, approx):
  Min: 9, Max: 4907, Mean: 438, Median: 363

GROUP-LEVEL STATISTICS (company-year)
Total groups: 132

Chunks per group:
  Min: 5, Max: 503, Mean: 110.2, Median: 86

Tokens per group (incl. prompt overhead):
  Min: 1219, Max: 231731, Mean: 48

9

In [ ]:
# pipeline.get_companies()  # Returns ['001', '003', '015', ...]


# can save
#df_barriers, df_motivators = pipeline.extract_company_data("001")
#pipeline.save_company_tables("001", df_barriers, df_motivators)

barriers, motivators = pipeline.extract_company_year("012", "2021") #small




Loading Ollama model: gemma3:4b


In [8]:
print(f"Barriers: {len(barriers)}, Motivators: {len(motivators)}")
for chunk_id, text in barriers:
    print(f"  [{chunk_id}] {text[:1000]}...")

Barriers: 11, Motivators: 1
  [0] In the first phase, which started in mid-2020, Dillinger and Saarstahl are already significantly reducing their emissions by optimising the existing blast furnace process....
  [1] This is done by blowing in coke oven gas and other highly hydrogen-containing gases....
  [2] An important project in this phase is "H2SYNgas", which is also designated as an IPCEI project....
  [3] At the end of their use cycle, steel products without loss of quality, can be recycled completely and as often as possible, and can be completely recycled into the economic cycle....
  [4] Moreover, the production of steel in Germany meets high standards in terms of environmental and climate protection in a global comparison....
  [5] Sustainability is also an important part of the ongoing transformation process....
  [6] The companies of the SHS Group want to produce CO2-neutrally up to 2045 steel at the latest....
  [7] With a joint sustainability report, the companies in the S

In [18]:
%timeit
# pipeline.inspect_pipeline_status()

pipeline.extract_all_companies()


EXTRACTING ALL COMPANIES
Total groups: 132
LLM context: 4096 tokens

Extracting: ArcelorMittal (001)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Groups: 13 | Avg chunks/group: 110.2


  001:   0%|          | 0/13 [00:00<?, ?it/s]

Loading Ollama model: qwen3:4b


  ✓ Extracted 62 barriers (8% matched), 47 motivators (21% matched)

Extracting: Acerinox (002)
Years: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 10 | Avg chunks/group: 110.2


  ✓ Extracted 32 barriers (50% matched), 73 motivators (22% matched)

Extracting: Outokumpu (003)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 12 | Avg chunks/group: 110.2


...

  ✓ Extracted 24 barriers (8% matched), 16 motivators (31% matched)

Extracting: Feralpi (014)
Years: ['2014', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 10 | Avg chunks/group: 110.2


  ✓ Extracted 57 barriers (5% matched), 36 motivators (6% matched)

Extracting: NipponSteel (015)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Groups: 13 | Avg chunks/group: 110.2


  ✓ Extracted 65 barriers (0% matched), 48 motivators (19% matched)

Extracting: Baosteel (016)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 12 | Avg chunks/group: 110.2


  ✓ Extracted 37 barriers (32% matched), 74 motivators (19% matched)

✓ EXTRACTION COMPLETE
  Companies: 15
  Total barriers: 748
  Total motivators: 673



{'001': (   company_id        company  year  \
  0         001  ArcelorMittal  2013   
  1         001  ArcelorMittal  2013   
  2         001  ArcelorMittal  2014   
  3         001  ArcelorMittal  2015   
  4         001  ArcelorMittal  2016   
  ..        ...            ...   ...   
  57        001  ArcelorMittal  2025   
  58        001  ArcelorMittal  2025   
  59        001  ArcelorMittal  2025   
  60        001  ArcelorMittal  2025   
  61        001  ArcelorMittal  2025   
  
                                               barriers source_chunk_id  \
  0   The main barrier to decarbonization in the ste...            None   
  1   This barrier is significant because the steel ...            None   
  2   Sorry, no text was provided. Please provide th...            None   
  3   Many European steel plants already face a shor...         001_102   
  4   Based on the provided text (which is the conca...            None   
  ..                                                ...     

# rag 2

In [ ]:
# from bertopic.representation import OpenAI,LlamaCPP
from nlp import TopicModelConfig, run_topic_modeling_pipeline

# Set True to ignore cached model/embeddings and retrain from scratch
FORCE_RETRAIN = True

config = TopicModelConfig(
    # Embedding model
    # embedding_model="sentence-transformers/all-mpnet-base-v2",
    embedding_model="BAAI/bge-small-en-v1.5",

    batch_size=64,

    # UMAP (dimensionality reduction)
    umap_n_neighbors=30,
    umap_n_components=15,
    umap_min_dist=0.05,
    umap_metric='cosine',
    umap_random_state=42,

    # HDBSCAN (clustering)
    hdbscan_min_cluster_size=10,  # Higher = fewer topics
    hdbscan_min_samples=2,        # Lower = less outliers
    hdbscan_metric='euclidean',
    hdbscan_cluster_selection_method='leaf',  # 'eom' (inclusive) or 'leaf' (tight, more cleanup required)

    # Vectorizer (c-TF-IDF)
    vectorizer_ngram_range=(1, 2),
    vectorizer_min_df=1,
    vectorizer_max_df=0.95,

    # Topic representation
    mmr_diversity=0.2, # 0 - pure relevance, redundant/simi word ... 1 - pure diverse. max diff word
    top_n_words=10,
    nr_topics=10,  # Set None for auto, or int to reduce post-hoc
    calculate_probabilities=True,

    # Outlier reduction (post-hoc)
    reduce_outliers=False,  # Reassign outliers to nearest topic
    reduce_outliers_strategy='embeddings',  # 'embeddings', 'c-tf-idf', or 'distributions'

    # Visualization UMAP (separate 2D projection)
    viz_umap_n_neighbors=10,
    viz_umap_n_components=2,
    viz_umap_min_dist=0.0,

    # LLM for topic labeling
    ollama_model="gemma3:4b",  # Fast + follows format. Avoid qwen3 (hidden thinking = slow)
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,

    # Misc
    verbose=True,
    embeddings_cache_path=None,
)

results = run_topic_modeling_pipeline(
    data_folder="../out",
    output_folder="../out/topics5",
    config=config,
    force_retrain=FORCE_RETRAIN
)
# Access results
barriers_df = results['barriers']['df']
motivators_df = results['motivators']['df']

In [ ]:
motivators_df